# Adoption Funnel Analysis: Multiple Methods for Product Insights

## What is Funnel Analysis?

A **funnel** is a visual representation of how users progress through a series of defined steps. In product analytics, we use it to track the customer journey—from first awareness through final conversion. The metaphor is apt: like a physical funnel, you pour water (users) in at the top, and progressively less comes out the bottom because some leaks out at each stage.

For example:
- **Landing Page** → **Sign Up** → **Product View** → **Add to Cart** → **Purchase**

## Why Do Product Teams Care?

1. **Identify Bottlenecks**: Where do most users drop off? If 80% complete signup but only 10% view the product, that's your biggest problem.
2. **Measure Conversion**: A single number—overall conversion rate—lets you compare campaigns, segments, or time periods.
3. **Segment Performance**: Do mobile users convert differently than desktop users? Do customers from social media behave differently than those from email?
4. **Set Benchmarks**: How long should it take a user to reach purchase? What's "normal"?

## This Notebook Covers 4 Methods

We'll explore **4 distinct approaches** to funnel analysis, each with different strengths and limitations:

1. **Classic Sequential Funnel** — The simplest: count users at each stage and compute conversion rates.
2. **Segmented Funnel Analysis** — Break the funnel by device and channel to spot hidden patterns.
3. **Time-to-Convert Analysis** — How long does each stage take?
4. **Weighted Funnel Scoring** — Assign importance to each stage and create a composite health score.

By the end, you'll understand when to use each method and how to combine insights for actionable product decisions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configure plotting
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Color palette (consistent throughout)
COLORS = {
    'primary': '#2E86AB',
    'secondary': '#F18F01',
    'tertiary': '#2CA58D',
    'accent': '#E15554'
}

# Paths
DATA_INPUT = Path('../data/inputs/funnel_clean.csv')
DATA_OUTPUT = Path('../data/outputs/nb02')
DATA_OUTPUT.mkdir(parents=True, exist_ok=True)

print(f"Input path: {DATA_INPUT}")
print(f"Output path: {DATA_OUTPUT}")

## Load & Explore Data

In [ ]:
# Load the data
df = pd.read_csv(DATA_INPUT)

# Display basic info
print("Dataset shape:", df.shape)
print("\nFirst few rows:")
print(df.head(10))
print("\nColumn dtypes:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
print("\nUnique values:")
for col in df.columns:
    print(f"  {col}: {df[col].nunique()} unique values")

In [ ]:
# Define the funnel stages in order
FUNNEL_STAGES = ['landing_page', 'signup', 'product_view', 'add_to_cart', 'purchase']

# Quick validation: confirm all stages are in the data
unique_stages = df['stage'].unique()
print("Stages in data:", sorted(unique_stages))
print("Expected funnel order:", FUNNEL_STAGES)

# Confirm 'reached' column (1 if reached, 0 if not)
print("\n'reached' column values:", df['reached'].unique())

## Method 1: Classic Sequential Funnel Analysis

### Overview

The **classic funnel** is the simplest and most widely used approach. Here's how it works:

1. **Count users at each stage**: How many unique users reached the landing page? How many completed signup? Etc.
2. **Calculate conversion rates**: What % of landing page visitors signed up? What % of signups viewed the product?
3. **Calculate overall conversion**: What % of all landing page visitors made a purchase?

### Benefits

- **Simple to compute and explain**: Anyone can understand a bar chart showing drop-off.
- **Fast iteration**: Run it in seconds to monitor health.
- **Industry standard**: Everyone knows how to interpret it.

### Limitations

- **Treats all users the same**: Doesn't account for segments (device, channel, geography).
- **Ignores time dynamics**: A user who took 6 months vs. 6 hours to convert both count the same.
- **Sequential assumption**: Assumes users follow a strict path (in reality, some may loop back).

### Use Case

Use the classic funnel as your **go-to check-in metric**. Run it weekly or daily to spot obvious problems.

In [ ]:
# Method 1: Classic Sequential Funnel
def classic_funnel(df, stages=FUNNEL_STAGES):
    """
    Compute classic funnel: unique user counts and conversion rates at each stage.
    Args:
        df: DataFrame with 'user_id' and 'stage' columns
        stages: list of stages in order
    Returns:
        DataFrame with user counts, conversion rates, and overall conversion
    """
    results = []
    total_users_at_stage_0 = df[df['stage'] == stages[0]]['user_id'].nunique()
    
    for i, stage in enumerate(stages):
        users_at_stage = df[df['stage'] == stage]['user_id'].nunique()
        
        # Stage-to-stage conversion: current stage / previous stage
        if i == 0:
            stage_to_stage_rate = 1.0  # First stage has 100%
        else:
            users_at_prev_stage = df[df['stage'] == stages[i-1]]['user_id'].nunique()
            stage_to_stage_rate = users_at_stage / users_at_prev_stage if users_at_prev_stage > 0 else 0
        
        # Overall conversion: current stage / first stage
        overall_rate = users_at_stage / total_users_at_stage_0 if total_users_at_stage_0 > 0 else 0
        
        results.append({
            'Stage': stage,
            'Users': users_at_stage,
            'Stage-to-Stage %': stage_to_stage_rate * 100,
            'Overall %': overall_rate * 100
        })
    
    return pd.DataFrame(results)

# Compute classic funnel
funnel_classic = classic_funnel(df)
print(funnel_classic.to_string(index=False))

# Save the results
funnel_classic.to_csv(DATA_OUTPUT / 'nb02_classic_funnel.csv', index=False)
print(f"\nSaved to {DATA_OUTPUT / 'nb02_classic_funnel.csv'}")

In [ ]:
# Visualize classic funnel
fig, ax = plt.subplots(figsize=(10, 6))

# Bar chart of user counts
stages = funnel_classic['Stage'].tolist()
users = funnel_classic['Users'].tolist()
colors = [COLORS['primary']] * len(stages)
bars = ax.bar(stages, users, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)

# Add value labels on bars
for i, (bar, val) in enumerate(zip(bars, users)):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(val)}\n({funnel_classic.iloc[i]["Stage-to-Stage %"]:.1f}%)',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xlabel('Funnel Stage', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Unique Users', fontsize=12, fontweight='bold')
ax.set_title('Classic Sequential Funnel: User Flow', fontsize=14, fontweight='bold', pad=20)
ax.set_ylim(0, max(users) * 1.15)
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(DATA_OUTPUT / 'nb02_01_classic_funnel.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"Chart saved to {DATA_OUTPUT / 'nb02_01_classic_funnel.png'}")

## Method 2: Segmented Funnel Analysis

### Overview

The classic funnel aggregates all users. **Segmented funnel analysis** breaks the funnel into groups based on user attributes:

- **By Device**: Mobile vs. Desktop vs. Tablet
- **By Channel**: Organic vs. Paid vs. Social vs. Email
- **By Geography**: Country, region, or continent
- **By Cohort**: Users acquired in the same week/month

This reveals whether certain segments have hidden bottlenecks.

### Benefits

- **Reveals hidden patterns**: Overall conversion might be 5%, but mobile might be 1% and desktop 8%.
- **Informs prioritization**: Focus engineering on the worst-performing segment first.
- **Tests hypotheses**: "Does our product work better on desktop?" Now you have data.

### Limitations

- **Smaller sample sizes**: Segments have fewer users, so noise is higher.
- **Multiple comparisons problem**: If you compare 10 segments, by chance 1 will look best and 1 worst.
- **Requires statistical testing**: Differences might not be significant. (We won't do formal stats here, but you should.)

### Use Case

Use segmented funnels when you want to **understand where your worst performers are** and **allocate resources to the highest-impact improvements**.

In [ ]:
# Method 2: Segmented Funnel Analysis
def segmented_funnel(df, segment_col, stages=FUNNEL_STAGES):
    """
    Compute funnel broken down by a segment column.
    Args:
        df: DataFrame with 'user_id', 'stage', and segment column
        segment_col: column name to segment by (e.g., 'device', 'channel')
        stages: list of stages in order
    Returns:
        DataFrame with funnel metrics for each segment
    """
    results = []
    for segment_value in sorted(df[segment_col].unique()):
        segment_df = df[df[segment_col] == segment_value]
        segment_funnel = classic_funnel(segment_df, stages)
        segment_funnel[segment_col] = segment_value
        results.append(segment_funnel)
    return pd.concat(results, ignore_index=True)

# Segment by device
print("=" * 60)
print("FUNNEL BY DEVICE")
print("=" * 60)
funnel_by_device = segmented_funnel(df, 'device')
print(funnel_by_device.to_string(index=False))
funnel_by_device.to_csv(DATA_OUTPUT / 'nb02_funnel_by_device.csv', index=False)

print("\n" + "=" * 60)
print("FUNNEL BY CHANNEL")
print("=" * 60)
funnel_by_channel = segmented_funnel(df, 'channel')
print(funnel_by_channel.to_string(index=False))
funnel_by_channel.to_csv(DATA_OUTPUT / 'nb02_funnel_by_channel.csv', index=False)

In [ ]:
# Visualize segmented funnel by device and channel\nfig, axes = plt.subplots(1, 2, figsize=(14, 5))\n\nstages_list = FUNNEL_STAGES\n\n# Device plot\nax = axes[0]\ndevices = sorted(df['device'].unique())\ncolors_device = ['#2E86AB', '#F18F01', '#2CA58D']\nx = np.arange(len(stages_list))\nwidth = 0.25\n\nfor idx, device in enumerate(devices):\n    device_data = funnel_by_device[funnel_by_device['device'] == device]\n    users_by_stage = device_data['Users'].values\n    ax.bar(x + idx * width, users_by_stage, width, label=device,\n           color=colors_device[idx], alpha=0.8, edgecolor='black', linewidth=0.8)\n\nax.set_xlabel('Funnel Stage', fontsize=11, fontweight='bold')\nax.set_ylabel('Number of Unique Users', fontsize=11, fontweight='bold')\nax.set_title('Funnel by Device Type', fontsize=12, fontweight='bold')\nax.set_xticks(x + width)\nax.set_xticklabels(stages_list, rotation=45, ha='right')\nax.legend()\nax.grid(axis='y', alpha=0.3)\n\n# Channel plot\nax = axes[1]\nchannels = sorted(df['channel'].unique())\ncolors_channel = ['#2E86AB', '#F18F01', '#2CA58D', '#E15554', '#6c63ff']\nx = np.arange(len(stages_list))\nwidth = 0.20\n\nfor idx, channel in enumerate(channels):\n    channel_data = funnel_by_channel[funnel_by_channel['channel'] == channel]\n    users_by_stage = channel_data['Users'].values\n    ax.bar(x + idx * width, users_by_stage, width, label=channel,\n           color=colors_channel[idx], alpha=0.8, edgecolor='black', linewidth=0.8)\n\nax.set_xlabel('Funnel Stage', fontsize=11, fontweight='bold')\nax.set_ylabel('Number of Unique Users', fontsize=11, fontweight='bold')\nax.set_title('Funnel by Channel', fontsize=12, fontweight='bold')\nax.set_xticks(x + width * 1.5)\nax.set_xticklabels(stages_list, rotation=45, ha='right')\nax.legend(fontsize=9)\nax.grid(axis='y', alpha=0.3)\n\nplt.tight_layout()\nplt.savefig(DATA_OUTPUT / 'nb02_02_segmented_funnel.png', dpi=300, bbox_inches='tight')\nplt.show()\nprint(f"Chart saved to {DATA_OUTPUT / 'nb02_02_segmented_funnel.png'}")

## Method 3: Time-to-Convert Analysis

### Overview

Users don't convert instantly. Some sign up on day 1, others take weeks. **Time-to-convert** analysis examines how long each stage takes:

1. For each user, calculate the time **between reaching a stage and the next stage**.
2. Summarize with median, mean, distribution (histogram).
3. Identify stages with unusually long gaps—those are friction points.

### Benefits

- **Sets realistic benchmarks**: "We should expect users to complete purchase within 3 days of adding to cart."
- **Identifies friction**: A stage with 30-day median time-to-next-stage is a pain point.
- **Guides UX prioritization**: Speed up slow stages first.
- **Detects anomalies**: If one cohort converts in 1 day vs. 30, something changed.

### Limitations

- **Right-censored data**: Users who *haven't* converted yet are excluded. This biases the data toward fast converters.
- **Requires timestamps**: Needs accurate, synchronized time data.
- **Excludes dropouts**: A user who quit at signup has "infinite time to next stage" but we can't measure it.

### Use Case

Use time-to-convert to **spot UX friction and set performance targets**.

In [ ]:
# Method 3: Time-to-Convert Analysis
# First, convert timestamp to datetime if needed
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Create a user journey: for each user, what's the time between consecutive stages?
def time_to_convert_analysis(df, stages=FUNNEL_STAGES):
    """
    For each stage, calculate time to reach the NEXT stage.
    Returns a dict: {stage: list of times in seconds}
    """
    results = {}
    
    # Group by user and get their stage history
    user_journeys = df.groupby('user_id').apply(
        lambda x: x.sort_values('timestamp'),
        include_groups=False
    )
    
    # For each stage, collect times until next stage
    for i, stage in enumerate(stages[:-1]):  # Don't include the last stage
        next_stage = stages[i + 1]
        times = []
        for user_id in df['user_id'].unique():
            user_data = df[df['user_id'] == user_id].sort_values('timestamp')
            # Find if user reached current stage and next stage
            reached_stage = user_data[user_data['stage'] == stage]
            reached_next = user_data[user_data['stage'] == next_stage]
            if len(reached_stage) > 0 and len(reached_next) > 0:
                # Get time of first occurrence of each stage
                time_at_stage = reached_stage.iloc[0]['timestamp']
                time_at_next = reached_next.iloc[0]['timestamp']
                # Only include if next stage is after current stage
                time_diff = (time_at_next - time_at_stage).total_seconds()
                if time_diff >= 0:
                    times.append(time_diff)
        results[f'{stage} → {next_stage}'] = times
    
    return results

time_to_convert = time_to_convert_analysis(df)

# Summarize
print("TIME TO NEXT STAGE (in hours, median and mean):")
print("=" * 70)
time_summary = []
for transition, times in time_to_convert.items():
    if len(times) > 0:
        times_hours = [t / 3600 for t in times]  # Convert to hours
        median = np.median(times_hours)
        mean = np.mean(times_hours)
        count = len(times)
        time_summary.append({
            'Transition': transition,
            'Count': count,
            'Median (hours)': median,
            'Mean (hours)': mean,
            'Min (hours)': min(times_hours),
            'Max (hours)': max(times_hours)
        })

time_summary_df = pd.DataFrame(time_summary)
print(time_summary_df.to_string(index=False))
time_summary_df.to_csv(DATA_OUTPUT / 'nb02_time_to_convert.csv', index=False)

In [ ]:
# Visualize time-to-convert distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

transitions = list(time_to_convert.keys())
for idx, transition in enumerate(transitions):
    ax = axes[idx]
    times = time_to_convert[transition]
    times_hours = [t / 3600 for t in times]  # Convert to hours
    
    # Histogram
    ax.hist(times_hours, bins=30, color=COLORS['primary'], alpha=0.7, edgecolor='black')
    
    # Add median line
    median = np.median(times_hours)
    ax.axvline(median, color=COLORS['accent'], linestyle='--', linewidth=2, label=f'Median: {median:.1f}h')
    ax.set_xlabel('Time (hours)', fontsize=10, fontweight='bold')
    ax.set_ylabel('Number of Users', fontsize=10, fontweight='bold')
    ax.set_title(transition, fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(DATA_OUTPUT / 'nb02_03_time_to_convert.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"Chart saved to {DATA_OUTPUT / 'nb02_03_time_to_convert.png'}")

## Method 4: Weighted Funnel Scoring

### Overview

In most businesses, **not all stages are equally important**. The purchase stage is more valuable than the landing page stage. A single metric—**funnel health score**—can capture overall segment performance.

**Steps:**

1. **Assign weights** to each stage based on business importance (e.g., revenue proximity, conversion difficulty).
2. **For each segment**, compute: sum of (users at stage × weight for that stage) / total users at first stage.
3. **Compare scores across segments**: 0 = worst, 100 = best. Higher is better.

Example weights:

- Landing Page: 1 (everyone starts here)
- Signup: 20 (meaningful engagement)
- Product View: 30 (deeper engagement)
- Add to Cart: 40 (intent to buy)
- Purchase: 100 (the goal)

### Benefits

- **Single number comparison**: Easy to see which segment is healthiest at a glance.
- **Accounts for business logic**: Weights let you encode "purchase is more important than signup."
- **Combines multiple insights**: A segment with low users but high stage can score well.

### Limitations

- **Weight assignment is subjective**: Different stakeholders may disagree on weights.
- **Hides details**: A high score might mask a severe drop-off at one stage.
- **Not causal**: A high score doesn't explain *why* a segment is healthy.

### Use Case

Use funnel scoring when you want to **rank segments and allocate effort** to the least healthy ones.

In [ ]:
# Method 4: Weighted Funnel Scoring
def weighted_funnel_score(df, segment_col, stages=FUNNEL_STAGES, weights=None):
    """
    Compute weighted funnel health score for each segment.
    Args:
        df: DataFrame
        segment_col: column to segment by
        stages: stages in order
        weights: dict mapping stage name to weight (default: linear escalation)
    Returns:
        DataFrame with segment and score
    """
    if weights is None:
        # Default: linear escalation (later stages more valuable)
        weights = {stage: (i + 1) * 20 for i, stage in enumerate(stages)}
    
    max_weight = max(weights.values())
    results = []
    
    for segment_value in sorted(df[segment_col].unique()):
        segment_df = df[df[segment_col] == segment_value]
        # Count users at each stage
        total_users_first_stage = segment_df[segment_df['stage'] == stages[0]]['user_id'].nunique()
        weighted_sum = 0
        for stage in stages:
            users_at_stage = segment_df[segment_df['stage'] == stage]['user_id'].nunique()
            weighted_sum += users_at_stage * weights[stage]
        
        # Normalize to 0-100 scale
        # Max possible: total_users_first_stage * max_weight
        max_possible = total_users_first_stage * max_weight
        score = (weighted_sum / max_possible * 100) if max_possible > 0 else 0
        
        results.append({
            segment_col: segment_value,
            'Funnel Health Score': score,
            'Users (First Stage)': total_users_first_stage
        })
    
    return pd.DataFrame(results)

# Define weights (later stages are more valuable)
weights = {
    'landing_page': 20,
    'signup': 30,
    'product_view': 40,
    'add_to_cart': 60,
    'purchase': 100
}

print("Funnel Weights (relative importance of each stage):")
for stage, weight in weights.items():
    print(f"  {stage}: {weight}")

print("\n" + "=" * 60)
print("FUNNEL HEALTH SCORE BY DEVICE")
print("=" * 60)
score_by_device = weighted_funnel_score(df, 'device', stages=FUNNEL_STAGES, weights=weights)
print(score_by_device.to_string(index=False))
score_by_device.to_csv(DATA_OUTPUT / 'nb02_score_by_device.csv', index=False)

print("\n" + "=" * 60)
print("FUNNEL HEALTH SCORE BY CHANNEL")
print("=" * 60)
score_by_channel = weighted_funnel_score(df, 'channel', stages=FUNNEL_STAGES, weights=weights)
print(score_by_channel.to_string(index=False))
score_by_channel.to_csv(DATA_OUTPUT / 'nb02_score_by_channel.csv', index=False)

In [ ]:
# Visualize weighted funnel scores
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Device scores
ax = axes[0]
devices = score_by_device['device'].tolist()
scores = score_by_device['Funnel Health Score'].tolist()
bars = ax.barh(devices, scores, color=[COLORS['primary'], COLORS['secondary'], COLORS['tertiary']],
               alpha=0.8, edgecolor='black', linewidth=1.5)
for i, (bar, score) in enumerate(zip(bars, scores)):
    width = bar.get_width()
    ax.text(width + 1, bar.get_y() + bar.get_height()/2., f'{score:.1f}',
            ha='left', va='center', fontsize=11, fontweight='bold')
ax.set_xlabel('Funnel Health Score (0-100)', fontsize=11, fontweight='bold')
ax.set_title('Weighted Funnel Score by Device', fontsize=12, fontweight='bold')
ax.set_xlim(0, 110)
ax.grid(axis='x', alpha=0.3)

# Channel scores
ax = axes[1]
channels = score_by_channel['channel'].tolist()
scores = score_by_channel['Funnel Health Score'].tolist()
colors_list = [COLORS['primary'], COLORS['secondary'], COLORS['tertiary'], COLORS['accent']]
bars = ax.barh(channels, scores, color=colors_list[:len(channels)],
               alpha=0.8, edgecolor='black', linewidth=1.5)
for i, (bar, score) in enumerate(zip(bars, scores)):
    width = bar.get_width()
    ax.text(width + 1, bar.get_y() + bar.get_height()/2., f'{score:.1f}',
            ha='left', va='center', fontsize=11, fontweight='bold')
ax.set_xlabel('Funnel Health Score (0-100)', fontsize=11, fontweight='bold')
ax.set_title('Weighted Funnel Score by Channel', fontsize=12, fontweight='bold')
ax.set_xlim(0, 110)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(DATA_OUTPUT / 'nb02_04_weighted_scores.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"Chart saved to {DATA_OUTPUT / 'nb02_04_weighted_scores.png'}")

## Method Comparison Table

| Method | Complexity | Data Required | Insight Depth | Best Use Case | Limitations |
|--------|-----------|------------------|-------|--------|---------|
| **Classic Sequential** | Low | User IDs, Stages | Surface-level | Health check-in, exec reporting | Treats all users the same; ignores segments and time |
| **Segmented Funnel** | Medium | + Device, Channel, or other attributes | Medium | Find best/worst performing segments | Smaller samples per segment; multiple comparison problem |
| **Time-to-Convert** | Medium | + Timestamps (accurate) | Medium | Identify friction points; set benchmarks | Right-censored data (dropouts not included); needs clean timestamps |
| **Weighted Scoring** | Medium-High | + Business logic (weights) | Medium | Rank segments; allocate resources | Weight assignment is subjective; hides stage-level details |

### When to Use Each

- **Just launched?** Start with **Classic Sequential**. Keep it simple.
- **Optimizing for growth?** Layer in **Segmented Funnel**. Find your worst performers.
- **Engineering roadmap decisions?** Use **Time-to-Convert**. Know where the friction is.
- **Board presentation?** Use **Weighted Scoring**. It's a single number executives understand.

## Key Findings & Product Recommendations

### Based on this analysis, here's what a product team should consider:

#### 1. **Identify the Biggest Drop-Off (Classic Funnel)**

Look at the stage-to-stage conversion rates. If you see a dramatic drop (e.g., 80% → 10%), *that's* your biggest opportunity.

**Action**: Form a task force to investigate and improve the worst stage. Run user interviews to understand the pain point.

#### 2. **Segment Your Audience (Segmented Funnel)**

Don't assume all segments behave the same. Compare device types and channels.

**Example Findings:**

- Mobile users have 3x higher drop-off at product view?  → Redesign mobile UX
- Email channel converts better than social? → Shift budget to email
- Android lags iPhone? → Debug Android-specific issues

**Action**: Prioritize improvements for the worst-performing segment first (biggest impact for effort).

#### 3. **Set Realistic Timelines (Time-to-Convert)**

If median time from signup to purchase is 5 days, don't expect users to buy on day 1. Design retention and engagement accordingly.

**Example Insights:**

- Users take 3 weeks to move from signup → product view? → Too much friction; simplify onboarding
- Time to purchase is highly variable (10 hours to 10 days)? → Some users are "fast deciders", others need nurture

**Action**: Create different journey paths for fast vs. slow converters. Add email nurture for the slow track.

#### 4. **Focus Effort Where It Matters (Weighted Scoring)**

Not all stages are created equal. If your business model shows purchase is 10x more important than signup, weight it that way.

**Action**: Allocate engineering resources proportional to stage importance. Fix the bottleneck in the highest-impact stage.

### Example Recommendations (Hypothetical)

If our analysis showed:

- Desktop converts at 12%, mobile at 3% → *Fix mobile UX*
- Organic users convert at 15%, paid at 5% → *Audit paid traffic quality or landing pages*
- Add-to-cart → Purchase takes 2 weeks → *Simplify checkout, add one-click purchase*

---

### Next Steps

1. **Validate findings** with quantitative tests (A/B tests) or qualitative research (user interviews).
2. **Pick one stage** to improve and measure the impact.
3. **Repeat monthly** to track progress.

## Guardrails & North Star: Where to Apply Them in Funnel Analysis

---

### North Star Connection

The funnel exists to drive a **North Star Metric**. Before building any funnel, ask: *"What is the single metric that best captures the value this product delivers?"*

For this dataset, a reasonable North Star is **Purchase Conversion Rate** (purchasers / landing page visitors). Every funnel stage is a lever that either helps or hurts that number.

**In a healthcare context (like SmarterDx):** the North Star might be *"Incremental diagnoses captured per hospital per month."* The funnel equivalent would be: Hospital Onboarded → Charts Reviewed by AI → AI Suggestions Generated → Suggestions Accepted by Physician → Diagnoses Coded.

**Interview tip:** When presented with a product idea, immediately ask *"What's the North Star?"* before diving into funnel mechanics. It shows you think about outcomes, not just activity.

---

### Guardrail Metrics for Funnel Optimization

When you optimize one part of the funnel, you can accidentally break something else. **Guardrails are metrics that should NOT get worse** while you improve your target metric.

Here's how guardrails apply to each funnel optimization:

| Funnel Optimization | What Could Go Wrong | Guardrail Metric |
|---|---|---|
| Increase signup rate | Lower-quality signups who never engage | % of signups who reach product_view within 7 days |
| Increase add-to-cart rate | Pushy UX leads to cart abandonment or returns | Return/refund rate, user satisfaction (NPS) |
| Increase purchase rate | Aggressive discounting erodes margin | Average order value, revenue per user |
| Speed up time-to-convert | Rushing users leads to buyer's remorse | 30-day retention rate, support ticket volume |

**In healthcare:** guardrails are even more critical. If you optimize for "more AI suggestions accepted," your guardrails must include:
- **False positive rate** — are the AI suggestions actually correct?
- **Physician workflow time** — does the tool make doctors slower?
- **Compliance flags** — are you introducing documentation errors?

**The rule of thumb:** For every metric you try to improve, define at least one guardrail that tells you if you're causing harm elsewhere.

---

### Rollback Criteria

In product analytics, you should define **when to stop** an optimization before it launches:
- If signup-to-engagement conversion drops below the baseline by more than 5%, pause the change
- If any guardrail metric degrades by more than 2 standard deviations from its historical mean, roll back immediately
- Set automated alerts on guardrail thresholds so the team doesn't have to monitor manually

**We can't measure these guardrails with our current dataset** (we'd need satisfaction surveys, error rates, and time-on-task logs), but knowing that you *should* define them is exactly what interviewers want to hear.

## Conclusion

Funnel analysis is **the most practical tool** in a product analyst's toolkit. It's simple, actionable, and directly tied to business outcomes.

The four methods we covered give you different perspectives:

1. **Classic funnel**: Quick health check
2. **Segmented funnel**: Find hidden problems
3. **Time-to-convert**: Identify friction
4. **Weighted scoring**: Rank priorities

**Best Practice**: Use all four together. Start with the classic funnel for overall health. Dive into segments to find the worst performer. Measure time-to-convert for that segment. Use weighted scores to decide resource allocation.

**Remember**: The goal isn't to have the highest conversion rate—it's to understand your users and build products they love. Funnel analysis is just one lens through which to see them.